# Use pymilvus query iterator to pull from database into dataframe

In [2]:
from pymilvus import MilvusClient
from dotenv import load_dotenv
import os
import pandas as pd
import re

load_dotenv(override=True)

True

In [5]:
file_type = "ipynb"
file_ext = f".{file_type}"

collection_name = "ipynb_skipclean_fixed250_tag30_prompt14"

In [10]:
fname_postfix = re.sub(fr"{file_type}_", "", collection_name)
fname_postfix = re.sub(r"_tag.*", "", fname_postfix)
fname_postfix

'skipclean_fixed250'

In [11]:
client = MilvusClient(
    uri = os.getenv("ZILLIZ_CLUSTER_ENDPOINT"),
    token = os.getenv("ZILLIZ_CLUSTER_TOKEN"),
)

In [12]:
client.load_collection(collection_name=collection_name)

In [ ]:
iterator = client.query_iterator(
    batch_size=10,
    collection_name=collection_name,
    output_fields=["*"],
)

2025-03-17 08:37:43,627 [WARNING][__setup_ts_by_request]: failed to get mvccTs from milvus server, use client-side ts instead (iterator.py:260)


In [ ]:
res = iterator.next()
res

In [ ]:
for batch in iterator:
    "Do something"

# Label by document name

In [96]:
import glob
from pymilvus import MilvusClient
from dotenv import load_dotenv
import os
import pandas as pd
import pickle
import re

load_dotenv(override=True)

file_type = "ipynb"
file_ext = f".{file_type}"

collection_name = "ipynb_skipclean_fixed250_tag30_prompt14"

files = glob.glob(f"Content/**/*{file_ext}", recursive=True)
files

['Content/Large Language Model ( LLM)/Part_3_Data_Preparation/Data_Preparation/Exercise_Solution_LLM_Data_Prep_Arxiv(WIP).ipynb',
 'Content/Large Language Model ( LLM)/Part_3_Data_Preparation/Data_Preparation/Exercise_LLM_Data_Prep.ipynb',
 'Content/Large Language Model ( LLM)/Part_3_Data_Preparation/Data_Preparation/Exercise_Solution_LLM_Data_Prep.ipynb',
 'Content/Large Language Model ( LLM)/Part_3_Data_Preparation/Data_Preparation/_m3.1-data-prep-text-lab-1.ipynb',
 'Content/Large Language Model ( LLM)/Part_3_Data_Preparation/Data_Preparation/Exercise_Solution_LLM_Data_Prep_for_Instruction_Tuning.ipynb',
 'Content/Large Language Model ( LLM)/Part_3_Data_Preparation/Data_Preparation/Demo_LLM_Data_Prep_101.ipynb',
 'Content/Large Language Model ( LLM)/Part_3_Data_Preparation/Data_Synthesization/Demo_Text_Data_Augmentation_Synthetic_Data.ipynb',
 'Content/Large Language Model ( LLM)/Part_3_Data_Preparation/Data_Synthesization/Exercise_Solution_Data_Augmentation_GPT.ipynb',
 'Content/La

In [97]:
client = MilvusClient(
    uri = os.getenv("ZILLIZ_CLUSTER_ENDPOINT"),
    token = os.getenv("ZILLIZ_CLUSTER_TOKEN"),
)
client.load_collection(collection_name)

In [98]:
fname_postfix = re.sub(f"{file_type}_", "", collection_name)
fname_postfix = re.sub(r"_tag.*", "", fname_postfix)
fname_postfix

'skipclean_fixed250'

In [99]:
os.makedirs(f"./labeling/{file_type}/", exist_ok=True)
os.makedirs(f"./labeling/{file_type}/{fname_postfix}", exist_ok=True) # save the chunks

d = {
    "id": [],
    "text": [],
    "filename": [],
    "label": [],
}
for file in files:
    head, tail = os.path.split(file)
    # res = client.query(
    #     collection_name=collection_name,
    #     filter=f'metadata["title"] == "{tail}"',
    #     output_fields=["id", "text", "metadata"]
    # )
    expr = 'metadata["title"] == {tail}'
    filter_params = {"tail": tail}
    res = client.query(
        collection_name=collection_name,
        filter=expr,
        filter_params=filter_params,
        output_fields=["id", "text", "metadata"]
    )

    if file_type == "pptx":
        if re.search("AWS", file):
            label = ["Infrastructure and Operations"]
        elif re.search("Lecture_LLM_Data_Preparation.pptx", file):
            label = ["Data Engineering", "Data Science"]
        elif re.search("LLM_Data_Collection.pptx", file):
            label = ["Data Engineering", "Data Science"]
        elif re.search("LLM_Data_Labelling.pptx", file):
            label = ["Data Science", "Machine Learning Engineering"]
        elif re.search("Copy of 10.2 Search Engine.pptx", file):
            label = ["Data Science", "Machine Learning Engineering"]
        elif re.search("Copy of m3.1-data-prep-timeseries-lecture.pptx", file):
            label = ["Data Science"]
        elif re.search("Large Language Model", file):
            label = ["Data Science", "Machine Learning Engineering"]
        elif re.search("Metabase", file):
            label = ["Data Analysis"]
    elif file_type == "ipynb":
        if re.search("Data wrangling with Python", file):
            # label = ["Data Analysis", "Data Engineering", "Data Science", "Machine Learning Engineering"]
            label = ["Data Analysis", "Data Science"]
        elif re.search("Data_Preparation/Demo_LLM_Data_Prep_101.ipynb", file):
            label = ["Data Engineering", "Data Science", "Machine Learning Engineering"]
        elif re.search("Data_Preparation/Exercise_Solution_LLM_Data_Prep_for_Instruction_Tuning.ipynb", file):
            label = ["Data Engineering", "Data Science", "Machine Learning Engineering"]
        elif re.search("Data_Annotation/Exercise_Solution_LLM_Annotation_Sagemaker_Groundtruth.ipynb", file):
            label = ["Data Science", "Infrastructure and Operations", "Machine Learning Engineering"]
        elif re.search("Data_Annotation", file):
            label = ["Data Science", "Machine Learning Engineering"]
        elif re.search("Data_Collection", file):
            label = ["Data Engineering", "Data Science"]
        elif re.search("Data_Synthesization", file):
            label = ["Data Science", "Machine Learning Engineering"]
        elif re.search("Data_Preparation", file):
            label = ["Data Engineering", "Data Science"]
        elif re.search("Part_9_RAG", file):
            label = ["Data Science", "Machine Learning Engineering"]
        elif re.search("Part_10_LLM_Use_Cases", file):
            label = ["Data Science", "Machine Learning Engineering"]

    for r in res:
        d["id"].append(r["id"])
        d["text"].append(r["text"])
        d["filename"].append(file)
        d["label"].append(label)

    for r in res:
        # fname_write = f"./labeling/pptx/{tail}-{r["id"]}.txt"
        fname_write = f"./labeling/{file_type}/{fname_postfix}/{r["id"]}.txt"
        to_write = f'ID: {r["id"]}\nFile: {file}\nGround Truth Label: ["{label}"]\n\n{r["text"]}'
        with open(fname_write, "w") as f:
            f.write(to_write)

df = pd.DataFrame.from_dict(d)
# df.to_pickle(f"./labeling/{file_type}/{file_type}_labels_{fname_postfix}.pkl")
# df.to_csv(f"./labeling/{file_type}/{file_type}_labels_{fname_postfix}.csv")

In [100]:
value_counts = df["filename"].value_counts().to_dict()

df["num_chunks"] = df["filename"].map(value_counts)
# df["num_chunks"].head()

In [101]:
print(len(df))
df_single_chunks = df.loc[df["num_chunks"] == 1]
df_multi_chunks = df.loc[df["num_chunks"] > 1]
print(len(df_single_chunks))
print(len(df_multi_chunks))

533
21
512


In [102]:
from sklearn.model_selection import train_test_split

train_idx, test_idx = train_test_split(list(range(len(df_multi_chunks))), test_size=0.2, random_state=0, stratify=df_multi_chunks["filename"])

In [103]:
train_idx = df_multi_chunks.index[train_idx]
test_idx = df_multi_chunks.index[test_idx]

In [105]:
df_multi_chunks.loc[train_idx, "split"] = "train"
df_multi_chunks.loc[test_idx, "split"] = "test"
# df_multi_chunks["split"].head(10)

In [106]:
df_single_chunks.loc[:, "split"] = ["train"]*len(df_single_chunks)

/tmp/ipykernel_158252/2650915547.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_single_chunks.loc[:, "split"] = ["train"]*len(df_single_chunks)


In [107]:
# pd.set_option('display.max_colwidth', 100)

df = pd.concat([df_multi_chunks, df_single_chunks])
print(len(df))
# df.head()

533


In [108]:
df.to_pickle(f"./labeling/{file_type}/{file_type}_{fname_postfix}_labels.pkl")
print(f"Saved ./labeling/{file_type}/{file_type}_{fname_postfix}_labels.pkl")
df.to_csv(f"./labeling/{file_type}/{file_type}_{fname_postfix}_labels.csv")
print(f"Saved ./labeling/{file_type}/{file_type}_{fname_postfix}_labels.csv")

Saved ./labeling/ipynb/ipynb_skipclean_fixed250_labels.pkl
Saved ./labeling/ipynb/ipynb_skipclean_fixed250_labels.csv


In [113]:
print(sum(df["split"] == "train"))
print(sum(df["split"] == "test"))

430
103


In [109]:
df_group = df.groupby(by=["filename", "split"]).count()

for file in df["filename"].values:
    print(file)
    train_samples = df_group.loc[file].loc["train"]["num_chunks"]
    try:
        test_samples = df_group.loc[file].loc["test"]["num_chunks"]
    except KeyError:
        test_samples = 0
    perc = round(train_samples/(train_samples+test_samples)*100,1)
    print(f"train/test/perc: {train_samples}/{test_samples}/{perc}")
    print("-"*50)
# df.groupby(by=["split", "filename"]).count()
# sanity_check.to_csv("sanity_check_split.csv")

Content/Large Language Model ( LLM)/Part_3_Data_Preparation/Data_Preparation/Exercise_Solution_LLM_Data_Prep_Arxiv(WIP).ipynb
train/test/perc: 7/2/77.8
--------------------------------------------------
Content/Large Language Model ( LLM)/Part_3_Data_Preparation/Data_Preparation/Exercise_Solution_LLM_Data_Prep_Arxiv(WIP).ipynb
train/test/perc: 7/2/77.8
--------------------------------------------------
Content/Large Language Model ( LLM)/Part_3_Data_Preparation/Data_Preparation/Exercise_Solution_LLM_Data_Prep_Arxiv(WIP).ipynb
train/test/perc: 7/2/77.8
--------------------------------------------------
Content/Large Language Model ( LLM)/Part_3_Data_Preparation/Data_Preparation/Exercise_Solution_LLM_Data_Prep_Arxiv(WIP).ipynb
train/test/perc: 7/2/77.8
--------------------------------------------------
Content/Large Language Model ( LLM)/Part_3_Data_Preparation/Data_Preparation/Exercise_Solution_LLM_Data_Prep_Arxiv(WIP).ipynb
train/test/perc: 7/2/77.8
------------------------------------

In [ ]:
res = client.query(
    collection_name=collection_name,
    filter='metadata["title"] == "Lecture_LLM_Data_Preparation.pptx"',
    output_fields=["id", "text", "metadata"]
)

res

data: ["{'id': '0db7212702ce7c51e56ff63796a7a04b07e7cf2e0064cc3e0414abf1acf0d710', 'text': 'Additionally, since most of the downstream tasks they evaluated on focused on English-language text, they used langdetect to filter out any pages that were not classified as English with a probability of at least 0.99\\nPreparing the C4 Dataset - 2\\x0cAutomatic filtering method (using WebText as a proxy for high quality documents), to improve the quality of Common Crawl (using a logistic regression based classifier).\\nFuzzy de-duplication of documents using a hashing method (MiniHashLSH)\\nPartial removal of text occurring in benchmark datasets that appears in CommonCrawl/WebText as well. “Unfortunately, a bug resulted in only partial removal of all detected overlaps from the training data. Due to the cost of training, it wasn’t feasible to retrain the model.”\\nPreparing the data for GPT-3\\nBrown et.al. (2020)\\x0cUses jusText on Web Archive files (raw HTTP responses including page HTML) for

In [34]:
for r in res:
    print(r["metadata"]["title"])

Lecture_LLM_Data_Preparation.pptx
Lecture_LLM_Data_Preparation.pptx
Lecture_LLM_Data_Preparation.pptx
Lecture_LLM_Data_Preparation.pptx
Lecture_LLM_Data_Preparation.pptx
Lecture_LLM_Data_Preparation.pptx
Lecture_LLM_Data_Preparation.pptx
Lecture_LLM_Data_Preparation.pptx
Lecture_LLM_Data_Preparation.pptx
Lecture_LLM_Data_Preparation.pptx
Lecture_LLM_Data_Preparation.pptx
Lecture_LLM_Data_Preparation.pptx
Lecture_LLM_Data_Preparation.pptx
Lecture_LLM_Data_Preparation.pptx
Lecture_LLM_Data_Preparation.pptx
Lecture_LLM_Data_Preparation.pptx
Lecture_LLM_Data_Preparation.pptx
Lecture_LLM_Data_Preparation.pptx
Lecture_LLM_Data_Preparation.pptx
Lecture_LLM_Data_Preparation.pptx
